# Supervised Fine-Tuning (SFT) with Serverless Customization on SageMaker AI

## Lab 2 — Fine-Tune with a Serverless LoRA Job

> **Prerequisite:** Complete **Lab 1** (1-prepare-data.ipynb1) before running this notebook. The 1contractnli-nda-review-train and contractnli-nda-review-val datasets must be registered in the SageMaker AI Registry.


| Lab | Notebook | What you'll do |
|-----|----------|----------------|
| **Lab 1** | 1-prepare-data.ipynb | Download ContractNLI, build training records, and register datasets |
| **Lab 2** | 2-fine-tune-llm.ipynb ← *you are here* | Submit a serverless LoRA fine-tuning job and register the result in the Model Registry |
| **Lab 3** | 3-evaluation.ipynb | Score the base, frontier, and fine-tuned model on held-out contracts |
| **Lab 4** | 4-deployment.ipynb | Deploy the merged model to a real-time SageMaker endpoint |

### What you'll do in this lab

1. **Select a base model** from SageMaker JumpStart — the starting point for fine-tuning
2. **Create a Model Package Group** in the SageMaker Model Registry — a versioned container for your fine-tuned models
3. **Configure the SFTTrainer** with LoRA and the datasets registered in Lab 1
4. **Inspect and override hyperparameters** — understanding the knobs that control fine-tuning quality
5. **Submit the serverless fine-tuning job** and track its progress

The fine-tuned model registered at the end of this lab becomes the artifact evaluated in Lab 3 and deployed in Lab 4 — using **LoRA** on SageMaker AI serverless customization, so there's no cluster to provision and no container to build.

### Step 1 – Choose a base model

The base model is configured in config.py, defaulting to NVIDIA Nemotron 3 Nano 30B (nvidia/nemotron-3-nano-30b-a3b), a mixture-of-experts model with 30B total parameters and ~3B active per forward pass, available through [SageMaker JumpStart](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-jumpstart.html). NVIDIA Nemotron 3 Super 120B is also available.

We fine-tune rather than prompt because worked examples made the small model measurably worse at detecting contradictions — the model's problem is reading the contract, not output format, and no number of examples fixes that. To try a different model, update BASE_MODEL_ID in config.py. The cell below lists all JumpStart models that support customization.


In [1]:
import boto3

from config import BASE_MODEL_ID

sm = boto3.client("sagemaker", region_name="us-east-1")
models, kwargs = [], {"HubName": "SageMakerPublicHub", "HubContentType": "Model",
                      "MaxResults": 100}
while True:
    r = sm.list_hub_contents(**kwargs)
    for item in r["HubContentSummaries"]:
        if "@capability:customization" in item.get("HubContentSearchKeywords", []):
            models.append(item["HubContentName"])
    if "NextToken" in r:
        kwargs["NextToken"] = r["NextToken"]
    else:
        break

print(f"base model for this lab: {BASE_MODEL_ID}\n")
print(f"customizable models available ({len(models)}):")
print("\n".join(sorted(models)))

base model for this lab: huggingface-reasoning-nvidia-nemotron-3-nano-30b-a3b-bf16

customizable models available (36):
deepseek-llm-r1-distill-llama-70b
deepseek-llm-r1-distill-llama-8b
deepseek-llm-r1-distill-qwen-1-5b
deepseek-llm-r1-distill-qwen-14b
deepseek-llm-r1-distill-qwen-32b
deepseek-llm-r1-distill-qwen-7b
huggingface-llm-nvidia-nemotron-3-super-120b-a12b-bf16
huggingface-llm-qwen2-5-14b-instruct
huggingface-llm-qwen2-5-32b-instruct
huggingface-llm-qwen2-5-72b-instruct
huggingface-llm-qwen2-5-7b-instruct
huggingface-reasoning-nvidia-nemotron-3-nano-30b-a3b-bf16
huggingface-reasoning-qwen3-06b
huggingface-reasoning-qwen3-1-7b
huggingface-reasoning-qwen3-14b
huggingface-reasoning-qwen3-32b
huggingface-reasoning-qwen3-4b
huggingface-reasoning-qwen3-8b
huggingface-vlm-gemma-4-26b-a4b-it
huggingface-vlm-gemma-4-31b-it
huggingface-vlm-gemma-4-e4b-it
huggingface-vlm-qwen3-5-27b
huggingface-vlm-qwen3-5-4b
huggingface-vlm-qwen3-5-9b
huggingface-vlm-qwen3-6-27b
meta-textgeneration-lla

In [ ]:
%load_ext autoreload
%autoreload 2

### Step 2 – Set up the SageMaker session

Re-establish the session, execution role, and the datasets registered in Lab 1. Fine-tuning output goes to the session's default S3 bucket.

In [ ]:
import os
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"  # pin ambient region: sagemaker.ai_registry.AIRHub
                                                  # hashes boto3's *default* session region into its
                                                  # private hub name, ignoring any sagemaker_session
                                                  # passed explicitly, so this must be set before import
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

boto_sess = boto3.Session() # remove region when complete
sess = Session(boto_session=boto_sess)
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(boto_session=boto_sess, default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

[08/24/26 11:41:19] INFO     Loading cached SSO token for nvsec-nv-aws-mpa                            ]8;id=279862;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=399776;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py#377\377]8;;\

sagemaker role arn: arn:aws:iam::492681118881:role/service-role/AmazonSageMaker-ExecutionRole-20201215T102238
sagemaker bucket: clockhart-bucket
sagemaker session region: us-east-1


In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from config import BASE_MODEL_ID, DATASET_PREFIX

base_model_id = BASE_MODEL_ID
training_dataset = DataSet.get(name=f"{DATASET_PREFIX}-train")
val_dataset = DataSet.get(name=f"{DATASET_PREFIX}-val")

output_path = (f"s3://{bucket_name}/{default_prefix}/{base_model_id}-contractnli"
               if default_prefix else f"s3://{bucket_name}/{base_model_id}-contractnli")
print(f"training output: {output_path}")

[08/24/26 11:41:29] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=222335;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=847418;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=901434;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=377330;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[08/24/26 11:41:30] INFO     Loading cached SSO token for nvsec-nv-aws-mpa                            ]8;id=626006;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=484014;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py#377\377]8;;\

[08/24/26 11:41:32] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=940315;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=122920;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Loading cached SSO token for nvsec-nv-aws-mpa                            ]8;id=57878;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=260494;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py#377\377]8;;\

...

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=610796;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=390339;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Loading cached SSO token for nvsec-nv-aws-mpa                            ]8;id=4017;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=400650;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py#377\377]8;;\

[08/24/26 11:41:36] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=71202;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=175515;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Loading cached SSO token for nvsec-nv-aws-mpa                            ]8;id=404077;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=740714;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py#377\377]8;;\

training output: s3://clockhart-bucket/huggingface-reasoning-nvidia-nemotron-3-nano-30b-a3b-bf16-contractnli


### Step 3 – Create a Model Package Group

A **Model Package Group** is a container in the [SageMaker Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html) that holds successive versions of a model — like a Git repository for your model weights, with each fine-tuning run adding a version and preserving history. It gives you lineage (which dataset, job, and hyperparameters produced each version), side-by-side comparison in the console, and a clean handoff: Lab 3 and Lab 4 retrieve the latest model from this group by name, without hardcoding ARNs.

In [ ]:
import hashlib

from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

# SageMaker caps Model Package Group names at 63 characters. Hash-truncate when
# base_model_id + suffix exceeds it, so notebooks 2, 3, 4 and 4a all derive the
# same name for any model id.
MAX_MPG_NAME_LENGTH = 63
suffix = "-contractnli-sft-mpg"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

print(f"Model Package Group: {model_package_group_name}")


[08/24/26 11:41:37] INFO     Runs on sagemaker prod, region:us-east-1                                  ]8;id=297891;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=208979;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#375\375]8;;\

                    INFO     Loading cached SSO token for nvsec-nv-aws-mpa                            ]8;id=758092;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=46050;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py#377\377]8;;\

Model Package Group already exists: huggingface-reasoning-nvidia-nemotro-b73b16-contractnli-sft-mpg


In [ ]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

try:
    ModelPackageGroup.get(model_package_group_name=model_package_group_name)
    print(f"already exists: {model_package_group_from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

try:
    ModelPackageGroup.get(model_package_group_name=model_package_group_name)
    print(f"already exists: {model_package_group_name}")
except ClientError:
    ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="ContractNLI NDA checklist review, serverless SFT",
    )
    print(f"created: {model_package_group_name}")name}")
except ClientError:
    ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="ContractNLI NDA checklist review, serverless SFT",
    )
    print(f"created: {model_package_group_name}")

### Step 4 – Configure the serverless SFT job

[Serverless model customization](https://docs.aws.amazon.com/sagemaker/latest/dg/serverless-customization.html) lets you submit a fine-tuning job without provisioning or managing GPU instances — you specify a base model, a technique, and a dataset, and SageMaker handles instance selection, distributed training, and registration. You pay only for compute actually consumed.

Serverless customization supports three techniques for both Nano 30B and Super 120B; SFT, RLVR and RLAIF. This workshop uses **SFT**, since the ContractNLI training records are exact labeled examples: given this contract, produce this specific JSON answer.

We use **LoRA** (TrainingType.LORA) rather than full fine-tuning: it freezes the base model's parameters and trains only small adapter matrices injected at the attention layers — a fraction of a percent of total parameters. That makes training far cheaper and faster, with comparable domain-adaptation quality. The adapter is merged back into the base weights at the end of training for efficient deployment (Lab 4). [Learn more](https://arxiv.org/abs/2106.09685).

> **Note:** `accept_eula=True` acknowledges the base model's end-user license agreement — required by SageMaker JumpStart before training can begin.

#### The SFTTrainer

SFTTrainer is a high-level SageMaker SDK class that abstracts away the infrastructure for a serverless SFT job — you describe what to fine-tune and the SDK handles the rest.

| Parameter | What it sets |
|-----------|-------------|
| `model` | The base model ID from SageMaker JumpStart |
| `training_type` | The fine-tuning method — `TrainingType.LORA` for LoRA adapters |
| `model_package_group` | The Model Registry group where the fine-tuned model will be registered |
| `training_dataset` / `validation_dataset` | The versioned datasets registered in Lab 1 |
| `s3_output_path` | S3 location for model artifacts and checkpoints |
| `accept_eula` | Acknowledges the base model's end-user license agreement |
| `base_job_name` | Prefix for the training job name |

Calling `.train(wait=False)` submits the job and returns immediately.

In [ ]:
from sagemaker.train.common import TrainingType
from sagemaker.train.sft_trainer import SFTTrainer

MAX_JOB_NAME_LENGTH, TIMESTAMP_LENGTH = 63, 15
base_job_name = "contractnli-sft"[: MAX_JOB_NAME_LENGTH - TIMESTAMP_LENGTH].rstrip("-")

trainer = SFTTrainer(
    model=base_model_id,
    training_type=TrainingType.LORA,
    model_package_group=model_package_group_name,
    training_dataset=training_dataset,
    validation_dataset=val_dataset,
    s3_output_path=output_path,
    sagemaker_session=sess,
    role=role,
    accept_eula=True,
    base_job_name=base_job_name,
)

print(f"fine-tuning:    {base_model_id}")
print(f"training type:  {TrainingType.LORA.value}")
print(f"model package:  {model_package_group_name}")
print(f"training data:  {training_dataset.name} v{training_dataset.version}")
print(f"validation:     {val_dataset.name} v{val_dataset.version}\n")

print("default hyperparameters:")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

default hyperparameters:
  global_batch_size: 128
  learning_rate: 0.0001
  lora_alpha: 16
  lora_rank: 128
  lr_scheduler: cosine
  lr_warmup_steps_ratio: 0.0
  max_epochs: 2
  min_lr: 1e-06
  mlflow_run_id: 
  mlflow_tracking_uri: 
  model_name_or_path: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
  name: example-name-5x0q0
  results_directory: 
  train_val_split_ratio: 1.0
  warmup_steps: 10
  weight_decay: 0.1


### Step 5 – Launch the training job

The cell below submits the job asynchronously (`wait=False`) and returns immediately. Training takes 20-25 minutes for 10 epochs on the 315 records that fit the sequence length window — a fixed part of that is SageMaker provisioning compute rather than training, so halving the epochs doesn't halve the wall-clock time.

Track progress either in the **AWS Console** (SageMaker AI > Training > Training jobs) or with the status-check and polling cells below.

The cell below sets six: the LoRA rank and alpha, the batch size, the epoch count, the
learning rate and the warmup ratio. The values are the ones the published run used.

Unlike the Qwen3 recipe, the Nemotron recipe exposes no `dataset_max_len`, so there is
no sequence cap to raise and no records are dropped for length. All 423 contracts train.

`global_batch_size` at 128 and `learning_rate` at 1e-4 already match the recipe defaults.

In [ ]:
trainer.hyperparameters.global_batch_size = 128
trainer.hyperparameters.max_epochs = 10
trainer.hyperparameters.learning_rate = 0.0001
trainer.hyperparameters.lr_warmup_steps_ratio = 0.1
trainer.hyperparameters.lora_rank = 32
trainer.hyperparameters.lora_alpha = 64

records = 423                         # one per contract, as written by notebook 1

gbs = int(trainer.hyperparameters.global_batch_size)
epochs = int(trainer.hyperparameters.max_epochs)
ratio = float(trainer.hyperparameters.lr_warmup_steps_ratio)

# Batches are carried across epoch boundaries rather than each epoch ending on a short
# one, so the step total is the floor of records x epochs / batch size.
steps = records * epochs // gbs

print(f"{records} records train")
print(f"{records / gbs:.1f} steps/epoch x {epochs} epochs = {steps} optimizer steps")
print(f"warmup = {ratio:.0%} = {round(ratio * steps)} steps\n")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")


315 of 423 records train (108 exceed dataset_max_len)
3 steps/epoch x 10 epochs = 30 optimizer steps
cosine schedule sized for 20 steps, warmup = 10% = 2 steps

  global_batch_size: 128
  learning_rate: 0.0001
  lora_alpha: 64
  lora_rank: 32
  lr_scheduler: cosine
  lr_warmup_steps_ratio: 0.1
  max_epochs: 10
  min_lr: 1e-06
  mlflow_run_id: 
  mlflow_tracking_uri: 
  model_name_or_path: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
  name: example-name-5x0q0
  results_directory: 
  train_val_split_ratio: 1.0
  warmup_steps: 10
  weight_decay: 0.1


### Launch

`wait=False` returns immediately. The job takes **25-30 minutes** for these 8 epochs.
A sizeable and fixed part of that is SageMaker provisioning the compute rather than
training, so the wall-clock does not scale with epochs the way you might expect:
halving the epochs does not halve the time.

In [8]:
training_job = trainer.train(wait=False)
TRAINING_JOB_NAME = training_job.training_job_name
print(f"launched: {TRAINING_JOB_NAME}")

[08/24/26 11:41:50] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=563474;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=824254;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[08/24/26 11:41:51] INFO     Role                                                          ]8;id=186598;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=889150;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             'arn:aws:iam::492681118881:role/service-role/AmazonSageMaker-                         
                             ExecutionRole-20201215T102238' validated for training. Using                          
                             it.                                                                                   

                    INFO     Training Job Name: contractnli-sft-20260824114151                   ]8;id=841071;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/train/sft_trainer.py\sft_trainer.py]8;;\:]8;id=328191;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/train/sft_trainer.py#331\331]8;;\

[08/24/26 11:41:52] INFO     Found 1 MLflow apps: [('mflowapp', 'Updated', '3.10.1')]         ]8;id=295335;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=944282;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py#213\213]8;;\

[08/24/26 11:41:53] INFO     Resolved MLflow app:                                             ]8;id=411850;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=931344;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py#236\236]8;;\
                             arn:aws:sagemaker:us-east-1:492681118881:mlflow-app/app-LU5W7ACY                      
                             LH65 (status: Updated, version: 3.10.1)                                               

                    INFO     MLflow resource ARN:                                             ]8;id=327088;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=919416;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py#988\988]8;;\
                             arn:aws:sagemaker:us-east-1:492681118881:mlflow-app/app-LU5W7ACY                      
                             LH65                                                                                  

[08/24/26 11:41:56] INFO     Auto-detecting whether dataset is multimodal: DataSet(               ]8;id=655555;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/train/common_utils/data_utils.py\data_utils.py]8;;\:]8;id=12353;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/train/common_utils/data_utils.py#140\140]8;;\
                               name='contractnli-nda-review-train',                                                
                               version='5.0.0',                                                                    
                               status='Available',                                                                 
                               method='generated',                                                                 
                               technique='sft',                                                                    
                               source='s3://sagemaker-us-east-1-492681118881/datasets/contractnli                  
                             -nda-review/train/dataset.jsonl',                                                     
                               created_time=datetime.datetime(2026, 8, 11, 9, 42, 32, 624000,                      
                             tzinfo=tzlocal()),                                                                    
                               updated_time=datetime.datetime(2026, 8, 11, 9, 42, 32, 624000,                      
                             tzinfo=tzlocal()),                                                                    
                               arn='arn:aws:sagemaker:us-east-1:492681118881:hub-content/HG9K321E                  
                             GJNQG6135NSG3KCSVVKN20MIK3MAFEJ25Q0IDRUE2NQG/DataSet/contractnli-nda                  
                             -review-train/5.0.0'                                                                  
                             )                                                                                     

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/clockhart/Library/Application Support/sagemaker/config.yaml


[08/24/26 11:42:01] INFO     Creating training_job resource.                                     ]8;id=340107;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=836281;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/resources.py#31239\31239]8;;\

launched: contractnli-sft-20260824114151


#### Check status

A one-off check of the training job's current status.

In [11]:
from sagemaker.core.resources import TrainingJob

response = TrainingJob.get(training_job_name=TRAINING_JOB_NAME)
print(f"Status: {response.training_job_status}")
print(f"Secondary: {response.secondary_status}")

Status: Completed
Secondary: Completed


#### *(Optional)* Poll until complete

Loops every 120 seconds and prints status until the job finishes or fails — use this if you'd rather stay in the notebook than switch to the console. Interrupt anytime with **Stop**; the job keeps running in the background regardless. Terminal statuses: `Completed`, `Failed`, `Stopped`. Continue to **notebook 3** once it reaches `Completed`.

In [10]:
import time

from sagemaker.core.resources import TrainingJob

while True:
    j = TrainingJob.get(training_job_name=TRAINING_JOB_NAME)
    print(f"{j.training_job_status} / {j.secondary_status}")
    if j.training_job_status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(120)

InProgress / Pending
InProgress / Downloading
InProgress / Downloading
InProgress / Training
InProgress / Training
InProgress / Training
InProgress / Training
InProgress / Training
InProgress / Training
InProgress / Training
InProgress / Training
InProgress / Uploading
InProgress / Uploading


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:10                                                                                   │
│                                                                                                  │
│    7 │   print(f"{j.training_job_status} / {j.secondary_status}")                                │
│    8 │   if j.training_job_status in ("Completed", "Failed", "Stopped"):                         │
│    9 │   │   break                                                                               │
│ ❱ 10 │   time.sleep(120)                                                                         │
│   11                                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
KeyboardInterrupt

### What you built

A LoRA fine-tuned model, registered as a new version in the model_package_group_name Model Package Group. Notebook 3 looks it up by this same name.

Continue to notebook 3 to evaluate it against the base and frontier models.